In [11]:
import numpy as np

class Gridworld:
    def __init__(self, shape=(5,5), start=(0,0), goal=(4,4), walls=None,
                 step_reward=-1, goal_reward=10, obstacle_reward=-10):
        self.shape = shape
        self.start = start
        self.goal = goal
        self.walls = walls if walls is not None else []
        self.current_state = start
        self.step_reward = step_reward
        self.goal_reward = goal_reward
        self.obstacle_reward = obstacle_reward
        self.is_terminal = False

        self.action_space = [0, 1, 2, 3]
        self.action_map = {0: 'Up', 1: 'Down', 2: 'Left', 3: 'Right'}

    def reset(self):
        self.current_state = self.start
        self.is_terminal = False
        return self.current_state

    def step(self, action):
        if self.is_terminal:
            return self.current_state, 0, True, {}

        row, col = self.current_state
        next_row, next_col = row, col
        reward = self.step_reward

        if action == 0:
            next_row = max(0, row - 1)
        elif action == 1:
            next_row = min(self.shape[0] - 1, row + 1)
        elif action == 2:
            next_col = max(0, col - 1)
        elif action == 3:
            next_col = min(self.shape[1] - 1, col + 1)

        if (next_row, next_col) in self.walls:
            self.current_state = (row, col)
        else:
            self.current_state = (next_row, next_col)

        if self.current_state == self.goal:
            reward = self.goal_reward
            self.is_terminal = True

        done = self.is_terminal
        info = {}
        return self.current_state, reward, done, info

    def get_all_states(self):
        states = []
        for r in range(self.shape[0]):
            for c in range(self.shape[1]):
                if (r,c) not in self.walls:
                    states.append((r,c))
        return states

    def display(self):
        grid = np.full(self.shape, ' . ')
        for r, c in self.walls:
            grid[r, c] = ' # '
        grid[self.start] = ' S '
        grid[self.goal] = ' G '
        grid[self.current_state] = ' A '
        for row in grid:
            print(''.join(row))
        print(f"Current state: {self.current_state}, Terminal: {self.is_terminal}")

In [5]:
def random_policy(state, env):

    return np.random.choice(env.action_space)


In [12]:
def generate_episode(env, policy):
    episode = []
    state = env.reset()
    done = False

    while not done:
        action = policy(state, env)
        next_state, reward, done, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state
    return episode

In [9]:
def monte_carlo_first_visit_prediction(env, policy, num_episodes, discount_factor=0.9):
    V = {state: 0 for state in env.get_all_states()}
    returns = {state: [] for state in env.get_all_states()}

    for i in range(num_episodes):
        episode = generate_episode(env, policy)
        states_in_episode = set()

        for t, (state, action, reward) in enumerate(episode):
            if state not in states_in_episode:
                states_in_episode.add(state)

                G = 0
                for k in range(t, len(episode)):
                    G += episode[k][2] * (discount_factor**(k - t))

                returns[state].append(G)
                V[state] = np.mean(returns[state])

    return V

In [ ]:
custom_walls = [(1,1), (1,2), (1,3), (2,1), (3,1), (3,2), (3,3)]
env = Gridworld(shape=(5,5), start=(0,0), goal=(4,4), walls=custom_walls, step_reward=-0.1, goal_reward=10)

print("Gridworld Layout:")
env.display()

num_episodes = 10000
discount_factor = 0.9

print(f"\nRunning Monte Carlo First-Visit Prediction for {num_episodes} episodes...")
estimated_V = monte_carlo_first_visit_prediction(env, random_policy, num_episodes, discount_factor)

print("\nEstimated State-Value Function (V(s)):")
value_grid = np.full(env.shape, np.nan)
for (r,c), value in estimated_V.items():
    value_grid[r,c] = value

for r in range(env.shape[0]):
    row_str = ""
    for c in range(env.shape[1]):
        if (r,c) in env.walls:
            row_str += "   #   "
        elif (r,c) == env.goal:
            row_str += f" {value_grid[r,c]:6.2f}"
        else:
            row_str += f" {value_grid[r,c]:6.2f}"
    print(row_str)

print(f"\nValue of the Start State {env.start}: {estimated_V[env.start]:.2f}")
print(f"Value of the Goal State {env.goal}: {estimated_V[env.goal]:.2f}")

Gridworld Layout:
 A  .  .  .  . 
 .  #  #  #  . 
 .  #  .  .  . 
 .  #  #  #  . 
 .  .  .  .  G 
Current state: (0, 0), Terminal: False

Running Monte Carlo First-Visit Prediction for 10000 episodes...
